# Real-Time Dangerous Animal Detection Using Big Data and Deep Learning with YOLO

- This project aims to develop a real-time dangerous animal detection system using YOLO (You Only Look Once) with deep learning and big data processing techniques.The system is designed to enhance public safety by detecting potentially hazardous animals such as snakes, lion, hyenas, leopard, elephant, fox, wild dog, wild boar and rhino through camera feeds. The dataset, consisting of both real-world and augmented images, is preprocessed using PySpark to ensure consistency in format, resolution, and quality. Various augmentation techniques, including motion blur, are applied to improve performance in dynamic environments. The model is trained with YOLOv11 to enhance detection accuracy under different conditions.

### `Step-1` : Gathering Dataset
- The dataset includes nine selected dangerous animal classes that are commonly seen in public areas: leopard, fox, hyena, lion, elephant, snake, wild dog, wild boar and rhino. To gather the dataset, images were sourced from `iNaturalist`. Approximately 2,000 images each were collected in total 18,000.

In [ ]:
import requests
import os
import time
from io import BytesIO
from PIL import Image

In [ ]:
# pip install requests pillow

### Download Images from iNaturalist API

In [ ]:
def download_inaturalist_images(species_names, images_per_species, save_dir='inaturalist_images', size=(1024, 1024)):
    os.makedirs(save_dir, exist_ok=True)
    base_url = 'https://api.inaturalist.org/v1/observations'

    for species in species_names:
        print(f"Searching images for: {species}")
        params = {
            'taxon_name': species,
            'per_page': min(images_per_species, 200),
            'page': 1,
            'order_by': 'created_at',
            'order': 'desc'
        }
        images_downloaded = 0
        page = 1
        
        while images_downloaded < images_per_species:
            params['page'] = page
            response = requests.get(base_url, params=params)
            
            if response.status_code != 200:
                print(f"Failed to retrieve data for {species}")
                break
                
            data = response.json()
            observations = data.get('results', [])
            
            if not observations:
                print(f"No more observations found for {species}")
                break
                
            for obs in observations:
                photo_urls = obs.get('photos', [])
                for photo in photo_urls:
                    if images_downloaded >= images_per_species:
                        break
                        
                    image_url = photo['url'].replace('square', 'medium')
                    
                    try:
                        img_response = requests.get(image_url)
                        img_response.raise_for_status()
                        img_data = BytesIO(img_response.content)
                        
                        with Image.open(img_data) as img:
                            # Convert RGBA to RGB if necessary
                            if img.mode in ('RGBA', 'LA', 'P'):
                                img = img.convert('RGB')
                            
                            img = img.resize(size, Image.LANCZOS)
                            
                            # Clean species name for filename
                            clean_species = species.replace(' ', '_').replace('/', '_')
                            filename = f"{clean_species}_{images_downloaded+1}.jpg"
                            filepath = os.path.join(save_dir, filename)
                            
                            img.save(filepath, format='JPEG', quality=95)
                            print(f"Downloaded and resized {filename}")
                            images_downloaded += 1
                            time.sleep(0.1)
                            
                    except Exception as e:
                        print(f"Error processing image: {e}")
                        continue
                        
            page += 1
            time.sleep(1)
            
        print(f"Finished downloading {images_downloaded} images for {species}\n")

# Define dangerous animal species to download
dangerous_animals = [
    'African Lion',
    'Leopard', 
    'Spotted Hyena',
    'African Elephant',
    'Venomous Snake',
    'African Wild Dog',
    'Wild Boar',
    'Black Rhinoceros',
    'Red Fox'
]

# Number of images to download per species
images_per_species = 2000

# Download images
download_inaturalist_images(dangerous_animals, images_per_species, save_dir='dangerous_animals_dataset')